In [8]:
import regex as re
gpt4_regex_pattern = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
gpt4_regex = re.compile(gpt4_regex_pattern)

In [ ]:
def get_recurrence(data: list):
    stats = dict()
    for j in data:
        for i in range(len(j)-1):
            pair = (j[i], j[i+1])
            stats[pair] = stats.get(pair,0) + 1
    return stats
with open('/Users/toprak/Documents/yz50/8. Hafta/nutuk.txt') as f:
    data = list(i.encode() for i in gpt4_regex.findall(f.read()))
print(data[:5])
stats = sorted(((v,k) for k,v in get_recurrence(data).items()), reverse=True)
data_merged = list(data)
iters = 20
i=0
merges = {}

while i<iters:
    new_data = list()
    merges[(stats[0][1][0], stats[0][1][1])] = 256+i
    for m in data_merged:
        j=0
        new_enc = list()
        while j < len(m):
            if j+1==len(m): new_enc.append(m[j]); j+=1
            elif m[j] == stats[0][1][0] and m[j+1] == stats[0][1][1]: new_enc.append(256+i); j+=2
            else: new_enc.append(m[j]); j+=1
        new_data.append(new_enc)
            
    print(f'{stats[0][1][0]} ({chr(stats[0][1][0]) if stats[0][1][0] < 256 else ""}) ve {stats[0][1][1]} ({chr(stats[0][1][1]) if stats[0][1][1] < 256 else ""}) birleştirildi. Yeni token: {256+i}')
    stats = sorted(((v,k) for k,v in get_recurrence(new_data).items()), reverse=True)

    data_merged = new_data
    i+=1
print(merges)
start_size = sum(len(m) for m in data)
end_size = sum(len(m) for m in data_merged)
print(f"Küçülme oranı: %{(1 - (end_size / start_size)) * 100:.2f}")

[b'TORK', b' DEVRlM', b' T', b' A', b' R']
32 ( ) ve 10 (
) birleştirildi. Yeni token: 256
105 (i) ve 110 (n) birleştirildi. Yeni token: 257
108 (l) ve 101 (e) birleştirildi. Yeni token: 258
97 (a) ve 110 (n) birleştirildi. Yeni token: 259
32 ( ) ve 109 (m) birleştirildi. Yeni token: 260
97 (a) ve 114 (r) birleştirildi. Yeni token: 261
105 (i) ve 114 (r) birleştirildi. Yeni token: 262
101 (e) ve 116 (t) birleştirildi. Yeni token: 263
32 ( ) ve 98 (b) birleştirildi. Yeni token: 264
100 (d) ve 101 (e) birleştirildi. Yeni token: 265
32 ( ) ve 118 (v) birleştirildi. Yeni token: 266
105 (i) ve 108 (l) birleştirildi. Yeni token: 267
105 (i) ve 121 (y) birleştirildi. Yeni token: 268
32 ( ) ve 116 (t) birleştirildi. Yeni token: 269
32 ( ) ve 105 (i) birleştirildi. Yeni token: 270
105 (i) ve 105 (i) birleştirildi. Yeni token: 271
97 (a) ve 116 (t) birleştirildi. Yeni token: 272
266 () ve 101 (e) birleştirildi. Yeni token: 273
97 (a) ve 108 (l) birleştirildi. Yeni token: 274
97 (a) ve 107 (k) bi

In [ ]:
def encode(data_input:str):
    regex_split = gpt4_regex.findall(data_input)
    regex_split = [i.encode(encoding='utf-8', errors='replace') for i in regex_split]
    skipping = False
    tokens = list()
    for spl in regex_split:
        encoded = list()
        for i in range(len(spl)):
            pair = (spl[i], spl[i+1]) if i<len(spl)-1 else (spl[i],)
            if pair in merges and not skipping: encoded.append(merges[pair]); skipping = True
            elif skipping: skipping = False
            else: encoded.append(pair[0])
        tokens.append(encoded)
    return tokens

vocab_merges = {v: k for k, v in merges.items()}
def decode(data_input: list):
    tokens_concat = list()
    for i in data_input:
        tokens = list(i)
        while any(t > 255 for t in tokens):
            new_tokens = []
            for t in tokens:
                if t > 255 and t in vocab_merges:
                    new_tokens.extend(vocab_merges[t])
                else:
                    new_tokens.append(t)
            tokens = new_tokens
        tokens_concat.append(tokens)

    return [bytes(i).decode('utf-8', errors='replace') for i in tokens_concat]

In [44]:
decode(encode("İzmir'den selamlar 😀"))

['İzmir', "'d", 'en', ' selamlar', ' 😀']